# Team profiles

Run/pass style, defense vs rush and pass, home/away form, and the current injury penalty for every NFL team.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from nfl_predictor.config import current_nfl_season
from nfl_predictor.features.profiles import team_profiles
from nfl_predictor.pipeline import build_feature_table

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

SEASON = current_nfl_season()
WEEK = None  # set to an int to freeze profiles as of that week
SEASON, WEEK

In [ ]:
features = build_feature_table()
profiles = team_profiles(features, season=SEASON, week=WEEK)
profiles

## Offensive style

Rush rate is the share of designed rushes vs passes. Teams about 5 points above the league average are **run-heavy**; 5 points below are **pass-heavy**.

In [ ]:
style_counts = profiles["style"].value_counts()
display(style_counts.to_frame("teams"))

ordered = profiles.sort_values("rush_rate")
colors = {"run-heavy": "#3b7d23", "balanced": "#6b7280", "pass-heavy": "#1d4ed8"}
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(ordered["team"], ordered["rush_rate"], color=ordered["style"].map(colors))
ax.axvline(ordered["rush_rate"].mean(), color="black", linestyle="--", linewidth=1)
ax.set_xlabel("Rush rate")
ax.set_title(f"{SEASON} offensive play mix")
plt.show()

## Defense vs run and pass

EPA allowed: **lower is better** for the defense.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 7), sharey=True)
rush = profiles.sort_values("def_rush_epa")
axes[0].barh(rush["team"], rush["def_rush_epa"], color="#92400e")
axes[0].set_title("EPA allowed vs rush")
axes[0].axvline(0, color="black", linewidth=1)

pas = profiles.sort_values("def_pass_epa")
axes[1].barh(pas["team"], pas["def_pass_epa"], color="#1d4ed8")
axes[1].set_title("EPA allowed vs pass")
axes[1].axvline(0, color="black", linewidth=1)
plt.tight_layout()
plt.show()

profiles.sort_values("off_epa", ascending=False)[
    ["team", "style", "off_epa", "rush_epa", "pass_epa", "def_rush_epa", "def_pass_epa"]
]

## Home / away and injuries

In [ ]:
split = profiles[["team", "home_pd", "road_pd", "form_pd"]].set_index("team")
fig, ax = plt.subplots(figsize=(10, 7))
split.sort_values("home_pd").plot.barh(ax=ax)
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("Point differential")
ax.set_title("Home vs road vs last-4 form")
plt.tight_layout()
plt.show()

profiles.sort_values("injury_penalty", ascending=False)[
    ["team", "injury_penalty", "starters_out", "qb_out", "style"]
]